# R6 - Selección de la combinación arquitectura CNN + representación de entrada (regla del IOV: AP)

**Candidatas (6 combinaciones operativas)**

- Arquitecturas: ResNet50, DenseNet121 y ConvNeXt-Tiny.
- Condiciones: **A** (imagen completa) y **B** (ROI automática obtenida con el segmentador U-Net fijo).
- La condición **C** (ROI de referencia) queda excluida: requiere la máscara de referencia, no está disponible en un uso real.

**Regla de selección (IOV aprobado)**

> "La configuración seleccionada es la de mayor AP media entre semillas, con criterio de desempate definido antes de ejecutar."

1. Criterio principal: mayor Average Precision (AP) media en `validation` (media de las 3 semillas).
2. Criterios complementarios, que se reportan pero no modifican la selección: ROC-AUC, sensibilidad, especificidad (umbral 0.5) y desviación estándar entre semillas.
3. Accuracy no se utiliza como criterio, por el desbalance de clases.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def locate_project_root() -> Path:
    current_dir = Path.cwd().resolve()

    for candidate in [current_dir, *current_dir.parents]:
        if (
            (candidate / "src").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "No se pudo localizar la raíz del proyecto."
    )


PROJECT_ROOT = locate_project_root()

## 1. Configuración y fuentes

Fuentes de resultados:

- R4: `results/classification/r4_benchmark/analysis/` (condición A) y, por corrida, `r4_benchmark/<arquitectura>/A/seed_<n>/` (predicciones, historial de entrenamiento y metadatos del checkpoint).
- R5: `results/classification/r5_input_representation/analysis/` (condiciones A, B y C; A se tomó de R4 sin reentrenar).
- Partición experimental: `configs/splits/mmotu_experimental_split.csv`, de la que solo se utilizan las filas de `validation` para comprobar la procedencia de las predicciones.

Todo se lee en modo solo lectura: no se entrenan modelos y no se utiliza `test`.

In [ ]:
CLASSIFICATION_DIR = PROJECT_ROOT / "results" / "classification"
R4_DIR = CLASSIFICATION_DIR / "r4_benchmark" / "analysis"
R5_DIR = CLASSIFICATION_DIR / "r5_input_representation" / "analysis"
OUTPUT_DIR = CLASSIFICATION_DIR / "r6_model_selection"
FIGURES_DIR = OUTPUT_DIR / "figures"
SPLIT_MANIFEST = PROJECT_ROOT / "configs" / "splits" / "mmotu_experimental_split.csv"

ARCHS = ["resnet50", "densenet121", "convnext_tiny"]
SELECTION_CONDITIONS = ["A", "B"]  # C excluida de la selección
SEEDS = [42, 43, 44]

PRIMARY_METRIC = "average_precision"
REPORTED_METRICS = ["average_precision", "roc_auc", "sensitivity", "specificity"]
THRESHOLD = 0.5
TARGET_SENSITIVITIES = [0.70, 0.80, 0.90]

ARCH_LABELS = {"resnet50": "ResNet50", "densenet121": "DenseNet121", "convnext_tiny": "ConvNeXt-Tiny"}
CONDITION_LABELS = {"A": "A (imagen completa)", "B": "B (ROI automática)"}
METRIC_LABELS = {
    "roc_auc": "ROC-AUC",
    "average_precision": "AP",
    "sensitivity": "Sensibilidad",
    "specificity": "Especificidad",
}
RUN_SOURCES = {"A": "r4_benchmark", "B": "r5_input_representation"}
CLASS_LABELS = ["Benigno", "Maligno"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

r4_summary = pd.read_csv(R4_DIR / "r4_architecture_summary.csv")
r4_per_seed = pd.read_csv(R4_DIR / "r4_metrics_per_seed.csv")
r5_summary = pd.read_csv(R5_DIR / "r5_condition_summary.csv")
r5_per_seed = pd.read_csv(R5_DIR / "r5_metrics_per_seed.csv")
r5_run_validation = pd.read_csv(R5_DIR / "r5_run_validation.csv")

# Solo se conservan las filas de validation de la partición (200 imágenes)
split_manifest = pd.read_csv(SPLIT_MANIFEST, usecols=["image_id", "binary_label", "experimental_split"])
validation_manifest = split_manifest[split_manifest["experimental_split"] == "validation"].set_index("image_id")
del split_manifest
assert len(validation_manifest) == 200

## 2. Verificación de los resultados consolidados

Antes de seleccionar se comprueba que:

1. las 27 corridas de R5 superaron las validaciones de protocolo, predicciones y métricas;
2. la media y la desviación estándar del resumen de R5 se reproducen a partir de los valores por semilla;
3. los valores de la condición A en R5 coinciden con los de R4 (por semilla, incluidos los IC 95 %).

In [ ]:
TOL = 1e-12

# 1. Validación de las 27 corridas en R5
assert len(r5_run_validation) == 27
for column in ["protocol_check", "predictions_check", "metrics_crosscheck"]:
    assert (r5_run_validation[column] == "ok").all(), column

# 2. Resumen de R5 reproducido desde los valores por semilla
recomputed = (
    r5_per_seed
    .melt(id_vars=["architecture", "condition", "seed"], value_vars=REPORTED_METRICS, var_name="metric", value_name="value")
    .groupby(["architecture", "condition", "metric"])["value"]
    .agg(mean_check="mean", std_check=lambda values: values.std(ddof=1), n_check="count")
    .reset_index()
)
check = r5_summary.merge(recomputed, on=["architecture", "condition", "metric"], how="inner", validate="one_to_one")
assert len(check) == len(ARCHS) * 3 * len(REPORTED_METRICS)
assert (check["n_check"] == 3).all()
assert (check["mean"] - check["mean_check"]).abs().max() < TOL
assert (check["std_between_seeds"] - check["std_check"]).abs().max() < TOL

# 3. Condición A: R5 coincide con R4
a_per_seed = r5_per_seed[r5_per_seed["condition"] == "A"].merge(
    r4_per_seed, on=["architecture", "seed"], suffixes=("_r5", "_r4"), validate="one_to_one"
)
assert len(a_per_seed) == len(ARCHS) * len(SEEDS)
for metric in REPORTED_METRICS:
    assert (a_per_seed[f"{metric}_r5"] - a_per_seed[f"{metric}_r4"]).abs().max() < TOL, metric

a_summary = r5_summary[r5_summary["condition"] == "A"].drop(columns="condition").merge(
    r4_summary, on=["architecture", "metric"], suffixes=("_r5", "_r4"), validate="one_to_one"
)
assert len(a_summary) == len(ARCHS) * len(REPORTED_METRICS)
for column in ["mean", "std_between_seeds", "ci95_lower", "ci95_upper"]:
    assert (a_summary[f"{column}_r5"] - a_summary[f"{column}_r4"]).abs().max() < TOL, column

print("Resultados consolidados verificados: R5 coherente con sus valores por semilla y condición A idéntica a R4")

## 3. Tabla de las seis combinaciones operativas

Valores de `validation`: media ± desviación estándar entre las 3 semillas. La tabla se ordena por AP media de mayor a menor, que es el criterio principal del IOV. Los IC 95 % son los bootstrap ya calculados en R4/R5; no se recalculan.

In [ ]:
candidates_long = r5_summary[r5_summary["condition"].isin(SELECTION_CONDITIONS)]
assert set(candidates_long["condition"]) == set(SELECTION_CONDITIONS)

candidates = candidates_long.pivot(
    index=["architecture", "condition"],
    columns="metric",
    values=["mean", "std_between_seeds", "ci95_lower", "ci95_upper"],
)
candidates.columns = [f"{metric}_{statistic}" for statistic, metric in candidates.columns]
candidates = candidates.reset_index()
assert len(candidates) == len(ARCHS) * len(SELECTION_CONDITIONS) == 6

candidates = candidates.sort_values(f"{PRIMARY_METRIC}_mean", ascending=False).reset_index(drop=True)
candidates.insert(0, "order_by_ap", np.arange(1, len(candidates) + 1))


def mean_sd(row, metric):
    return f"{row[f'{metric}_mean']:.3f} ± {row[f'{metric}_std_between_seeds']:.3f}"


compact_table = pd.DataFrame(
    {
        "Arquitectura": candidates["architecture"].map(ARCH_LABELS),
        "Condición": candidates["condition"].map(CONDITION_LABELS),
        "AP media ± DE": candidates.apply(mean_sd, axis=1, metric="average_precision"),
        "ROC-AUC media ± DE": candidates.apply(mean_sd, axis=1, metric="roc_auc"),
        "Sensibilidad media ± DE": candidates.apply(mean_sd, axis=1, metric="sensitivity"),
        "Especificidad media ± DE": candidates.apply(mean_sd, axis=1, metric="specificity"),
        "IC 95 % AP": candidates.apply(
            lambda row: f"[{row['average_precision_ci95_lower']:.3f}, {row['average_precision_ci95_upper']:.3f}]", axis=1
        ),
    }
)

candidates.to_csv(OUTPUT_DIR / "r6_candidates.csv", index=False)
compact_table.to_csv(OUTPUT_DIR / "r6_candidates_compact.csv", index=False)

compact_table

## 4. Aplicación del criterio principal

Se selecciona la combinación con mayor AP media. Se verifica que no exista un empate exacto en AP.

In [ ]:
max_primary = candidates[f"{PRIMARY_METRIC}_mean"].max()
tied = candidates[candidates[f"{PRIMARY_METRIC}_mean"] == max_primary]
assert len(tied) == 1, "Empate exacto en AP media"
selected = tied.iloc[0]

runner_up = candidates[
    ~((candidates["architecture"] == selected["architecture"]) & (candidates["condition"] == selected["condition"]))
].iloc[0]

print(
    "Combinación que satisface el criterio principal: "
    f"{ARCH_LABELS[selected['architecture']]} + {CONDITION_LABELS[selected['condition']]} "
    f"(AP media = {selected['average_precision_mean']:.4f})"
)
print("No se produjo un empate exacto en AP; no se aplicó ningún criterio de desempate.")

### 4.1 Verificaciones de la selección

Se comprueba que la combinación seleccionada es ConvNeXt-Tiny con A, que su AP media coincide con `r4_architecture_summary.csv`, que no hubo empate exacto en AP y que las predicciones guardadas de cada semilla corresponden a las 200 imágenes de `validation` (22 malignas), sin ninguna imagen de `test`.

In [ ]:
selected_arch, selected_condition = selected["architecture"], selected["condition"]


def combination_label(row):
    return f"{ARCH_LABELS[row['architecture']]} + {CONDITION_LABELS[row['condition']]}"


def run_dir(arch, condition, seed):
    return CLASSIFICATION_DIR / RUN_SOURCES[condition] / arch / condition / f"seed_{seed}"


def load_validation_predictions(arch, condition, seed):
    run_path = run_dir(arch, condition, seed) / "val_predictions_best_auc.csv"
    predictions = pd.read_csv(run_path)
    assert len(predictions) == 200 and predictions["image_id"].is_unique, run_path
    assert int(predictions["y_true"].sum()) == 22, run_path
    # Todas las imágenes pertenecen a validation (por tanto, ninguna es de test) y las etiquetas coinciden con la partición
    assert predictions["image_id"].isin(validation_manifest.index).all(), run_path
    manifest_labels = validation_manifest.loc[predictions["image_id"], "binary_label"].to_numpy()
    assert (predictions["y_true"].to_numpy() == manifest_labels).all(), run_path
    return predictions


# 1. Combinación seleccionada
assert (selected_arch, selected_condition) == ("convnext_tiny", "A")

# 2. AP media idéntica a la de r4_architecture_summary.csv
r4_selected_ap = r4_summary[(r4_summary["architecture"] == selected_arch) & (r4_summary["metric"] == PRIMARY_METRIC)]
assert len(r4_selected_ap) == 1
assert abs(selected[f"{PRIMARY_METRIC}_mean"] - r4_selected_ap["mean"].iloc[0]) < TOL

# 3. Sin empate exacto en AP
assert len(tied) == 1

# 4. Predicciones de la combinación seleccionada: 200 imágenes, 22 malignas, solo validation
selected_predictions = {seed: load_validation_predictions(selected_arch, selected_condition, seed) for seed in SEEDS}

print("Verificaciones superadas: ConvNeXt-Tiny + A, AP coherente con R4, sin empate exacto y predicciones de validation (200 imágenes, 22 malignas)")

## 5. Comparación de las dos combinaciones con mayor AP media

Se comparan la combinación seleccionada y la segunda según el criterio principal, con los valores ya consolidados.

In [ ]:
comparison_rows = []
for metric in REPORTED_METRICS:
    comparison_rows.append(
        {
            "Métrica": METRIC_LABELS[metric],
            "Seleccionada: media ± DE": mean_sd(selected, metric),
            "Seleccionada: IC 95 %": f"[{selected[f'{metric}_ci95_lower']:.3f}, {selected[f'{metric}_ci95_upper']:.3f}]",
            "Segunda: media ± DE": mean_sd(runner_up, metric),
            "Segunda: IC 95 %": f"[{runner_up[f'{metric}_ci95_lower']:.3f}, {runner_up[f'{metric}_ci95_upper']:.3f}]",
            "Diferencia de medias (seleccionada − segunda)": f"{selected[f'{metric}_mean'] - runner_up[f'{metric}_mean']:+.3f}",
        }
    )

top_two_comparison = pd.DataFrame(comparison_rows)

print(
    f"Seleccionada: {ARCH_LABELS[selected['architecture']]} + {CONDITION_LABELS[selected['condition']]}\n"
    f"Segunda:      {ARCH_LABELS[runner_up['architecture']]} + {CONDITION_LABELS[runner_up['condition']]}"
)

ap_overlap = not (
    selected["average_precision_ci95_lower"] > runner_up["average_precision_ci95_upper"]
    or runner_up["average_precision_ci95_lower"] > selected["average_precision_ci95_upper"]
)
print("Los IC 95 % de AP se superponen:", "sí" if ap_overlap else "no", "(descriptivo, no es una prueba)")

top_two_comparison

## 6. Figura comparativa

AP media (3 semillas) e IC 95 % bootstrap de las seis combinaciones operativas, con las arquitecturas en orden fijo y las condiciones A y B una junto a otra. La combinación seleccionada se identifica con marcador relleno.

In [ ]:
CONDITION_OFFSETS = {"A": -0.12, "B": 0.12}
CONDITION_MARKERS = {"A": "o", "B": "s"}

rows = candidates.set_index(["architecture", "condition"])
positions = np.arange(len(ARCHS))

fig, ax = plt.subplots(figsize=(7.0, 4.4))

for condition in SELECTION_CONDITIONS:
    for position, arch in zip(positions, ARCHS):
        row = rows.loc[(arch, condition)]
        x = position + CONDITION_OFFSETS[condition]
        is_selected = (arch == selected["architecture"]) and (condition == selected["condition"])

        ax.vlines(x, row["average_precision_ci95_lower"], row["average_precision_ci95_upper"], color="#4a4a4a", linewidth=1.6, zorder=2)
        ax.hlines([row["average_precision_ci95_lower"], row["average_precision_ci95_upper"]], x - 0.04, x + 0.04,
                  color="#4a4a4a", linewidth=1.6, zorder=2)
        ax.plot(
            x, row["average_precision_mean"], linestyle="none", marker=CONDITION_MARKERS[condition], markersize=8,
            markerfacecolor="#2b2b2b" if is_selected else "white", markeredgecolor="#2b2b2b", zorder=3,
        )

    ax.plot([], [], linestyle="none", marker=CONDITION_MARKERS[condition], markerfacecolor="white",
            markeredgecolor="#2b2b2b", label=CONDITION_LABELS[condition])

ax.plot([], [], linestyle="none", marker="o", markerfacecolor="#2b2b2b", markeredgecolor="#2b2b2b",
        label="Combinación seleccionada")

ax.set_xticks(positions)
ax.set_xticklabels([ARCH_LABELS[arch] for arch in ARCHS])
ax.set_xlim(-0.6, len(ARCHS) - 0.4)
ax.set_ylim(0, 1)
ax.set_ylabel("AP (validation)")
ax.set_title("AP de las combinaciones operativas\nmedia de 3 semillas e IC 95 % bootstrap", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="upper right")
ax.grid(axis="y", color="#dddddd", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "r6_ap_candidates_ci95.png", dpi=300, bbox_inches="tight")
plt.show()

### 6.1 Combinación seleccionada frente a la segunda

Valores consolidados de las tres semillas (punto = media; barra = IC 95 % bootstrap de R4/R5) para las dos combinaciones con mayor AP media. **AP es el criterio principal de selección**; ROC-AUC, sensibilidad y especificidad se muestran como criterios complementarios y no modifican la selección.

In [ ]:
COMPARISON_STYLES = {
    "selected": {"offset": -0.13, "marker": "o", "face": "#2b2b2b"},
    "runner_up": {"offset": 0.13, "marker": "s", "face": "white"},
}
METRIC_ROLE = {
    "average_precision": "criterio principal",
    "roc_auc": "complementaria",
    "sensitivity": "complementaria",
    "specificity": "complementaria",
}

fig, ax = plt.subplots(figsize=(8.0, 4.4))
positions = np.arange(len(REPORTED_METRICS))

# Fondo que distingue el criterio principal (primera métrica de REPORTED_METRICS)
assert REPORTED_METRICS[0] == PRIMARY_METRIC
ax.axvspan(-0.5, 0.5, color="#efefef", zorder=0)

for key, row in (("selected", selected), ("runner_up", runner_up)):
    style = COMPARISON_STYLES[key]
    for position, metric in zip(positions, REPORTED_METRICS):
        x = position + style["offset"]
        lower, upper = row[f"{metric}_ci95_lower"], row[f"{metric}_ci95_upper"]
        ax.vlines(x, lower, upper, color="#4a4a4a", linewidth=1.6, zorder=2)
        ax.hlines([lower, upper], x - 0.04, x + 0.04, color="#4a4a4a", linewidth=1.6, zorder=2)
        ax.plot(x, row[f"{metric}_mean"], linestyle="none", marker=style["marker"], markersize=8,
                markerfacecolor=style["face"], markeredgecolor="#2b2b2b", zorder=3)
    prefix = "Seleccionada" if key == "selected" else "Segunda"
    ax.plot([], [], linestyle="none", marker=style["marker"], markerfacecolor=style["face"],
            markeredgecolor="#2b2b2b", label=f"{prefix}: {combination_label(row)}")

ax.set_xticks(positions)
ax.set_xticklabels([f"{METRIC_LABELS[metric]}\n({METRIC_ROLE[metric]})" for metric in REPORTED_METRICS])
ax.set_xlim(-0.5, len(REPORTED_METRICS) - 0.5)
ax.set_ylim(0, 1)
ax.set_ylabel("Valor en validation")
ax.set_title("Combinación seleccionada frente a la segunda\nmedia de 3 semillas e IC 95 % bootstrap", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="lower left")
ax.grid(axis="y", color="#dddddd", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "r6_selected_vs_runnerup_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

### 6.2 Matrices de confusión de la combinación seleccionada

Matrices de confusión en `validation` de cada una de las tres semillas de la combinación seleccionada (ConvNeXt-Tiny + A), calculadas desde las predicciones guardadas del checkpoint de cada corrida con el umbral fijo 0.5. No se elige una semilla representativa ni se combinan probabilidades entre semillas: la figura muestra la variabilidad del comportamiento al umbral entre las tres ejecuciones. Los recuentos se contrastan con los de `r5_metrics_per_seed.csv`.

In [ ]:
confusion = {}

for seed in SEEDS:
    predictions = selected_predictions[seed]

    y_true = predictions["y_true"].to_numpy()
    y_pred = (predictions["probability_malignant"].to_numpy() >= THRESHOLD).astype(int)

    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())

    # Contraste con los resultados consolidados de R4/R5
    consolidated = r5_per_seed[
        (r5_per_seed["architecture"] == selected_arch)
        & (r5_per_seed["condition"] == selected_condition)
        & (r5_per_seed["seed"] == seed)
    ]
    assert len(consolidated) == 1
    assert (tn, fp, fn, tp) == tuple(int(consolidated[k].iloc[0]) for k in ["tn", "fp", "fn", "tp"]), seed

    confusion[seed] = np.array([[tn, fp], [fn, tp]])

fig, axes = plt.subplots(1, len(SEEDS), figsize=(11.0, 3.8))

for ax, seed in zip(axes, SEEDS):
    matrix = confusion[seed]
    ax.imshow(matrix, cmap="Greys", vmin=0, vmax=200)

    cell_names = [["TN", "FP"], ["FN", "TP"]]
    for i in range(2):
        for j in range(2):
            value = matrix[i, j]
            ax.text(j, i, f"{cell_names[i][j]}\n{value}", ha="center", va="center", fontsize=11,
                    color="white" if value > 100 else "black")

    sensitivity = matrix[1, 1] / matrix[1].sum()
    specificity = matrix[0, 0] / matrix[0].sum()

    ax.set_xticks([0, 1])
    ax.set_xticklabels(CLASS_LABELS)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(CLASS_LABELS)
    ax.set_xlabel(f"Predicción (umbral {THRESHOLD})\nsens. {sensitivity:.3f} | espec. {specificity:.3f}", fontsize=9)
    ax.set_title(f"Seed {seed}", fontsize=11)

axes[0].set_ylabel("Clase real")
fig.suptitle(
    f"Matrices de confusión en validation: {combination_label(selected)}\n(178 benignas, 22 malignas)",
    fontsize=11,
)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "r6_selected_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Regla aplicada y criterios alternativos

La regla aplicada es la del protocolo de R6 y la del IOV aprobado: mayor AP media en `validation` entre semillas. Con ella se selecciona ConvNeXt-Tiny + A. No se produjo un empate exacto en AP; no se aplicó ningún criterio de desempate.

El criterio ROC-AUC media se evaluó como análisis alternativo (sección 8.1): con él se habría seleccionado DenseNet121 + A. Ese análisis es descriptivo y no modifica la selección. Los resultados de un análisis previo con el criterio ROC-AUC se conservan como material histórico en `results/classification/r6_model_selection/historico/`.

## 8. Análisis de sensibilidad (no modifica la selección)

Las siguientes comprobaciones son descriptivas. Ninguna se utiliza para seleccionar: la selección es la de la sección 4.

### 8.1 Ranking por ROC-AUC media

Se reordenan las seis combinaciones por ROC-AUC media para mostrar qué se habría seleccionado con ese criterio alternativo.

In [ ]:
roc_ranking = candidates.sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
roc_ranking.insert(0, "order_by_roc_auc", np.arange(1, len(roc_ranking) + 1))

roc_top = roc_ranking.iloc[0]
assert (roc_top["architecture"], roc_top["condition"]) == ("densenet121", "A")

roc_ranking_table = pd.DataFrame(
    {
        "Orden por ROC-AUC": roc_ranking["order_by_roc_auc"],
        "Orden por AP": roc_ranking["order_by_ap"],
        "Arquitectura": roc_ranking["architecture"].map(ARCH_LABELS),
        "Condición": roc_ranking["condition"].map(CONDITION_LABELS),
        "ROC-AUC media ± DE": roc_ranking.apply(mean_sd, axis=1, metric="roc_auc"),
        "AP media ± DE": roc_ranking.apply(mean_sd, axis=1, metric="average_precision"),
    }
)

print(
    f"Con ROC-AUC media como criterio se habría seleccionado {combination_label(roc_top)} "
    f"(ROC-AUC {roc_top['roc_auc_mean']:.4f}); con AP media, {combination_label(selected)} "
    f"(AP {selected['average_precision_mean']:.4f})."
)

roc_ranking_table

### 8.2 Diferencia de AP entre las dos primeras frente a la variabilidad entre semillas

Se compara la diferencia de AP media entre la seleccionada y la segunda con la desviación estándar entre semillas de cada una.

In [ ]:
ap_gap = selected["average_precision_mean"] - runner_up["average_precision_mean"]
sd_selected = selected["average_precision_std_between_seeds"]
sd_runner_up = runner_up["average_precision_std_between_seeds"]

print(f"Diferencia de AP media (seleccionada − segunda): {ap_gap:+.4f}")
print(f"DE entre semillas de la AP: seleccionada {sd_selected:.4f}; segunda {sd_runner_up:.4f}")
print(f"Razón diferencia / menor DE: {abs(ap_gap) / min(sd_selected, sd_runner_up):.3f}")

practical_tie = abs(ap_gap) < min(sd_selected, sd_runner_up)
if practical_tie:
    print("La diferencia es menor que la DE entre semillas de ambas combinaciones: se interpreta como empate práctico.")
else:
    print("La diferencia no es menor que la DE entre semillas de ambas combinaciones.")
print("Esta lectura es descriptiva: la selección se mantiene según la regla del IOV (mayor AP media).")

### 8.3 Especificidad a sensibilidades fijas (exploratorio, post hoc)

**Análisis exploratorio y post hoc, definido después de conocer los resultados; no se utiliza para seleccionar.** Para ConvNeXt-Tiny-A y DenseNet121-A se calcula, por semilla y en media, la especificidad en `validation` cuando el umbral se fija en el valor más alto que alcanza al menos la sensibilidad objetivo (0.70, 0.80 y 0.90). Con 22 casos malignos la sensibilidad solo toma valores múltiplos de 1/22, por lo que la sensibilidad alcanzada puede superar ligeramente el objetivo; se reporta junto con el umbral. Los umbrales se eligen sobre el mismo `validation` en que se mide la especificidad, de modo que los valores son optimistas y solo sirven para describir.

In [ ]:
def specificity_at_sensitivity(y_true, scores, target):
    positives = np.sort(scores[y_true == 1])[::-1]
    k = int(np.ceil(target * len(positives) - 1e-9))
    threshold = positives[k - 1]
    y_pred = scores >= threshold
    achieved_sensitivity = float(y_pred[y_true == 1].mean())
    specificity = float((~y_pred[y_true == 0]).mean())
    return float(threshold), achieved_sensitivity, specificity


EXPLORATORY_COMBINATIONS = [("convnext_tiny", "A"), ("densenet121", "A")]

exploratory_predictions = {
    (arch, condition, seed): load_validation_predictions(arch, condition, seed)
    for arch, condition in EXPLORATORY_COMBINATIONS
    for seed in SEEDS
}

fixed_rows = []
for (arch, condition, seed), predictions in exploratory_predictions.items():
    y_true = predictions["y_true"].to_numpy()
    scores = predictions["probability_malignant"].to_numpy()
    for target in TARGET_SENSITIVITIES:
        threshold, achieved, specificity = specificity_at_sensitivity(y_true, scores, target)
        fixed_rows.append(
            {
                "analysis": "fixed_sensitivity",
                "architecture": arch,
                "condition": condition,
                "seed": seed,
                "target_sensitivity": target,
                "threshold": threshold,
                "achieved_sensitivity": achieved,
                "specificity": specificity,
            }
        )

fixed_sensitivity = pd.DataFrame(fixed_rows)
assert (fixed_sensitivity["achieved_sensitivity"] >= fixed_sensitivity["target_sensitivity"] - 1e-9).all()

fixed_means = (
    fixed_sensitivity
    .groupby(["analysis", "architecture", "condition", "target_sensitivity"], as_index=False)[["threshold", "achieved_sensitivity", "specificity"]]
    .mean()
    .assign(seed="mean")
)

fixed_table = fixed_sensitivity.pivot_table(
    index=["architecture", "target_sensitivity"], columns="seed", values="specificity"
)
fixed_table.columns = [f"Espec. seed {seed}" for seed in fixed_table.columns]
fixed_table["Espec. media"] = fixed_table.mean(axis=1)
fixed_table["Sens. alcanzada media"] = fixed_sensitivity.pivot_table(
    index=["architecture", "target_sensitivity"], values="achieved_sensitivity"
)["achieved_sensitivity"]
fixed_table = fixed_table.reset_index()
fixed_table.insert(0, "Arquitectura", fixed_table.pop("architecture").map(ARCH_LABELS))
fixed_table = fixed_table.rename(columns={"target_sensitivity": "Sensibilidad objetivo"})

fixed_table.round(3)

### 8.4 Mediana de la probabilidad asignada a los casos malignos

Para documentar la calibración de ConvNeXt-Tiny-A (en particular en las semillas 43 y 44) se calcula, por semilla, la mediana de la probabilidad de malignidad que el modelo asigna a los 22 casos malignos de `validation`. Una mediana por debajo de 0.5 implica que al menos la mitad de los casos malignos queda bajo el umbral fijo. Se incluye DenseNet121-A como referencia.

In [ ]:
median_rows = []
for (arch, condition, seed), predictions in exploratory_predictions.items():
    y_true = predictions["y_true"].to_numpy()
    scores = predictions["probability_malignant"].to_numpy()
    median_rows.append(
        {
            "analysis": "median_probability_malignant",
            "architecture": arch,
            "condition": condition,
            "seed": seed,
            "median_probability_malignant": float(np.median(scores[y_true == 1])),
            "median_probability_benign": float(np.median(scores[y_true == 0])),
            "sensitivity_at_threshold": float((scores[y_true == 1] >= THRESHOLD).mean()),
        }
    )

median_probability = pd.DataFrame(median_rows)

median_table = median_probability.pivot_table(
    index="architecture", columns="seed", values="median_probability_malignant"
)
median_table.columns = [f"Mediana seed {seed}" for seed in median_table.columns]
median_table.index = median_table.index.map(ARCH_LABELS)
median_table.index.name = "Arquitectura"

for arch in [selected_arch]:
    below = median_probability[
        (median_probability["architecture"] == arch) & (median_probability["median_probability_malignant"] < THRESHOLD)
    ]["seed"].tolist()
    print(f"{ARCH_LABELS[arch]}-A: semillas con mediana de probabilidad en malignos < {THRESHOLD}: {below if below else 'ninguna'}")

sensitivity_analysis = pd.concat([fixed_sensitivity, fixed_means, median_probability], ignore_index=True)
sensitivity_analysis.to_csv(OUTPUT_DIR / "r6_sensitivity_analysis.csv", index=False)

median_table.round(3)

## 9. Curvas de entrenamiento de la combinación seleccionada

Se leen los `training_history.json` de ConvNeXt-Tiny + A (semillas 42, 43 y 44) de `r4_benchmark/`, sin reentrenar. En cada curva se marca la época del checkpoint, que es la de máxima `val_roc_auc` según `best_auc_meta.json` (el early stopping se basa en `val_weighted_loss`, con paciencia 5). La pérdida graficada es la ponderada por clase que registra el historial. No se utiliza accuracy.

In [ ]:
HISTORY_KEYS = ["train_loss", "val_loss", "val_roc_auc", "val_average_precision"]

histories, checkpoint_epochs, curve_rows = {}, {}, []

for seed in SEEDS:
    seed_dir = CLASSIFICATION_DIR / "r4_benchmark" / selected_arch / selected_condition / f"seed_{seed}"
    with open(seed_dir / "training_history.json", encoding="utf-8") as f:
        history = json.load(f)
    with open(seed_dir / "best_auc_meta.json", encoding="utf-8") as f:
        best_meta = json.load(f)
    with open(seed_dir / "run_config.json", encoding="utf-8") as f:
        run_config = json.load(f)

    n_epochs = len(history["train_loss"])
    assert n_epochs == run_config["epochs_run"], seed
    for key in HISTORY_KEYS:
        assert len(history[key]) == n_epochs, (seed, key)

    # Las épocas se numeran desde 1; el checkpoint es el máximo de val_roc_auc
    epoch_best = best_meta["epoch"]
    assert epoch_best == run_config["epoch_best"], seed
    index_best = epoch_best - 1
    assert abs(history["val_roc_auc"][index_best] - max(history["val_roc_auc"])) < TOL, seed
    assert abs(history["val_roc_auc"][index_best] - best_meta["val_roc_auc"]) < TOL, seed
    assert abs(history["val_loss"][index_best] - best_meta["val_weighted_loss"]) < TOL, seed

    # Primera época desde la cual val_loss permanece por encima de train_loss hasta el final de la corrida
    persistent_gap_epoch = None
    for index in range(n_epochs - 1, -1, -1):
        if history["val_loss"][index] > history["train_loss"][index]:
            persistent_gap_epoch = index + 1
        else:
            break

    histories[seed] = history
    checkpoint_epochs[seed] = epoch_best
    curve_rows.append(
        {
            "seed": seed,
            "epochs_run": n_epochs,
            "checkpoint_epoch": epoch_best,
            "early_stopped": bool(run_config["early_stopped"]),
            "early_stopped_at_epoch": run_config.get("early_stopped_at_epoch"),
            "train_loss_last": history["train_loss"][-1],
            "val_loss_last": history["val_loss"][-1],
            "train_loss_checkpoint": history["train_loss"][index_best],
            "val_loss_checkpoint": history["val_loss"][index_best],
            "val_roc_auc_checkpoint": history["val_roc_auc"][index_best],
            "val_average_precision_checkpoint": history["val_average_precision"][index_best],
            "min_val_loss_epoch": int(np.argmin(history["val_loss"])) + 1,
            "persistent_gap_from_epoch": persistent_gap_epoch,
        }
    )

curves_summary = pd.DataFrame(curve_rows)
curves_summary.to_csv(OUTPUT_DIR / "r6_curvas_resumen.csv", index=False)

curves_table = pd.DataFrame(
    {
        "Semilla": curves_summary["seed"],
        "Épocas ejecutadas": curves_summary["epochs_run"],
        "Época del checkpoint": curves_summary["checkpoint_epoch"],
        "Early stopping": curves_summary.apply(
            lambda row: f"sí (época {row['early_stopped_at_epoch']})" if row["early_stopped"] else "no", axis=1
        ),
        "train_loss última": curves_summary["train_loss_last"].round(3),
        "val_loss última": curves_summary["val_loss_last"].round(3),
        "train_loss checkpoint": curves_summary["train_loss_checkpoint"].round(3),
        "val_loss checkpoint": curves_summary["val_loss_checkpoint"].round(3),
        "Época de mínimo val_loss": curves_summary["min_val_loss_epoch"],
        "val_loss > train_loss hasta el final desde la época": curves_summary["persistent_gap_from_epoch"],
    }
)

curves_table

In [ ]:
SEED_STYLES = {
    42: {"color": "#2b2b2b", "linestyle": "-"},
    43: {"color": "#5c5c5c", "linestyle": "--"},
    44: {"color": "#8c8c8c", "linestyle": ":"},
}


def plot_curve(ax, seed, key):
    style = SEED_STYLES[seed]
    values = histories[seed][key]
    epochs = np.arange(1, len(values) + 1)
    ax.plot(epochs, values, color=style["color"], linestyle=style["linestyle"], linewidth=1.6, label=f"Seed {seed}")
    best = checkpoint_epochs[seed]
    ax.plot(best, values[best - 1], linestyle="none", marker="o", markersize=8,
            markerfacecolor=style["color"], markeredgecolor="#2b2b2b", zorder=3)


def style_curve_axis(ax, title, ylabel):
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Época")
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.grid(axis="y", color="#dddddd", linewidth=0.7, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


checkpoint_handle = plt.Line2D([], [], linestyle="none", marker="o", markersize=8, markerfacecolor="#2b2b2b",
                               markeredgecolor="#2b2b2b", label="Época del checkpoint (máx. val_roc_auc)")

# Figura 1: pérdida de entrenamiento y de validation
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.2), sharey=True)
for seed in SEEDS:
    plot_curve(axes[0], seed, "train_loss")
    plot_curve(axes[1], seed, "val_loss")
style_curve_axis(axes[0], "Pérdida de entrenamiento", "Pérdida ponderada")
style_curve_axis(axes[1], "Pérdida de validation", "Pérdida ponderada")
handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(handles + [checkpoint_handle], labels + [checkpoint_handle.get_label()], frameon=False, fontsize=9, loc="upper left")
fig.suptitle(f"Pérdida por época: {combination_label(selected)}", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "r6_curvas_entrenamiento_loss.png", dpi=300, bbox_inches="tight")
plt.show()

# Figura 2: AP y ROC-AUC en validation
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.2), sharey=True)
for seed in SEEDS:
    plot_curve(axes[0], seed, "val_average_precision")
    plot_curve(axes[1], seed, "val_roc_auc")
style_curve_axis(axes[0], "AP en validation", "Valor en validation")
style_curve_axis(axes[1], "ROC-AUC en validation", "Valor en validation")
axes[0].set_ylim(0, 1)
handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(handles + [checkpoint_handle], labels + [checkpoint_handle.get_label()], frameon=False, fontsize=9, loc="lower right")
fig.suptitle(f"Métricas de validation por época: {combination_label(selected)}", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "r6_curvas_entrenamiento_metricas.png", dpi=300, bbox_inches="tight")
plt.show()

for row in curve_rows:
    gap = row["persistent_gap_from_epoch"]
    gap_text = f"desde la época {gap}" if gap is not None else "no de forma sostenida hasta el final"
    print(
        f"Seed {row['seed']}: val_loss permanece por encima de train_loss {gap_text}; "
        f"checkpoint en la época {row['checkpoint_epoch']} de {row['epochs_run']}; mínimo de val_loss en la época {row['min_val_loss_epoch']}."
    )

### 9.1 Lectura descriptiva de las curvas

- La tabla y las líneas impresas indican, para cada semilla, desde qué época `val_loss` queda por encima de `train_loss` hasta el final de la corrida. Es una descripción de las curvas, no una prueba de sobreajuste.
- El checkpoint se eligió por máxima `val_roc_auc`, no por pérdida; por eso su época no coincide necesariamente con el mínimo de `val_loss` (columna "Época de mínimo val_loss").
- Las curvas provienen de una sola corrida por semilla y de un `validation` con 22 casos malignos; AP y ROC-AUC por época fluctúan en esa escala. Los datos no permiten atribuir la separación entre pérdidas a una causa concreta.

## 10. Registro formal de la selección

Se guarda un registro con la combinación seleccionada, la regla aplicada (incluida la referencia al IOV y el criterio alternativo evaluado), sus valores y las fuentes utilizadas.

In [ ]:
def metric_record(row):
    return {
        metric: {
            "mean": float(row[f"{metric}_mean"]),
            "std_between_seeds": float(row[f"{metric}_std_between_seeds"]),
            "ci95_lower": float(row[f"{metric}_ci95_lower"]),
            "ci95_upper": float(row[f"{metric}_ci95_upper"]),
        }
        for metric in REPORTED_METRICS
    }


selection_record = {
    "selected": {
        "architecture": selected["architecture"],
        "condition": selected["condition"],
        "metrics_validation": metric_record(selected),
    },
    "runner_up": {
        "architecture": runner_up["architecture"],
        "condition": runner_up["condition"],
        "metrics_validation": metric_record(runner_up),
    },
    "selection_rule": {
        "primary_criterion": "max mean Average Precision on validation (mean of 3 seeds)",
        "iov_reference": "La configuración seleccionada es la de mayor AP media entre semillas, con criterio de desempate definido antes de ejecutar.",
        "complementary_criteria": ["roc_auc", "std_between_seeds", "sensitivity", "specificity"],
        "excluded_criteria": ["accuracy (class imbalance)"],
        "candidate_conditions": SELECTION_CONDITIONS,
        "excluded_conditions": {"C": "requires reference mask; idealized condition in R5"},
        "candidate_architectures": ARCHS,
        "seeds": SEEDS,
        "threshold": THRESHOLD,
    },
    "sensitivity_criterion": {
        "metric": "mean ROC-AUC",
        "would_select": f"{roc_top['architecture']}-{roc_top['condition']}",
    },
    "data": {
        "evaluation_split": "validation",
        "n_validation": 200,
        "n_malignant_validation": 22,
        "test_used": False,
        "new_training": False,
    },
    "sensitivity_analysis": {
        "file": str((OUTPUT_DIR / "r6_sensitivity_analysis.csv").relative_to(PROJECT_ROOT)),
        "status": "exploratory and post hoc; not used for selection",
        "ap_gap_selected_minus_runner_up": float(ap_gap),
        "ap_gap_below_between_seed_sd": bool(practical_tie),
    },
    "sources": [
        str((R4_DIR / "r4_architecture_summary.csv").relative_to(PROJECT_ROOT)),
        str((R4_DIR / "r4_metrics_per_seed.csv").relative_to(PROJECT_ROOT)),
        str((R5_DIR / "r5_condition_summary.csv").relative_to(PROJECT_ROOT)),
        str((R5_DIR / "r5_metrics_per_seed.csv").relative_to(PROJECT_ROOT)),
        str((R5_DIR / "r5_run_validation.csv").relative_to(PROJECT_ROOT)),
        "results/classification/r4_benchmark/convnext_tiny/A/seed_*/val_predictions_best_auc.csv",
        "results/classification/r4_benchmark/convnext_tiny/A/seed_*/training_history.json",
    ],
}

with open(OUTPUT_DIR / "r6_selection.json", "w", encoding="utf-8") as f:
    json.dump(selection_record, f, indent=2, ensure_ascii=False)

print(json.dumps(selection_record["selected"], indent=2, ensure_ascii=False))

In [ ]:
s, r = selected, runner_up
print("Razón de la selección")
print("-" * 70)
print(
    f"Se selecciona {ARCH_LABELS[s['architecture']]} con {CONDITION_LABELS[s['condition']]} porque presenta la mayor "
    f"AP media en validation entre las seis combinaciones operativas "
    f"({s['average_precision_mean']:.3f} ± {s['average_precision_std_between_seeds']:.3f}; IC 95 % "
    f"[{s['average_precision_ci95_lower']:.3f}, {s['average_precision_ci95_upper']:.3f}]), según el IOV aprobado."
)
print(
    f"La segunda combinación es {ARCH_LABELS[r['architecture']]} con {CONDITION_LABELS[r['condition']]} "
    f"(AP {r['average_precision_mean']:.3f} ± {r['average_precision_std_between_seeds']:.3f}; diferencia de medias "
    f"{s['average_precision_mean'] - r['average_precision_mean']:+.4f}, menor que la DE entre semillas de ambas: "
    f"{'sí' if practical_tie else 'no'})."
)
print(
    f"Criterios complementarios de la seleccionada: ROC-AUC {mean_sd(s, 'roc_auc')}, "
    f"sensibilidad {mean_sd(s, 'sensitivity')}, especificidad {mean_sd(s, 'specificity')}."
)
print("La selección se realizó solo con validation; test no se ha utilizado.")

## 11. Consideraciones

- La selección aplica la regla del IOV aprobado a resultados ya consolidados; no se entrenaron modelos, no se recalcularon ROI ni bootstrap y no se utilizó `test`. El criterio ROC-AUC se evaluó solo como análisis alternativo (sección 8.1).
- Los valores de `validation` tienen un sesgo optimista común, porque el checkpoint de cada corrida de R4/R5 se seleccionó por máxima ROC-AUC en ese mismo conjunto, y la selección de R6 se realiza también sobre `validation`. La estimación no sesgada del desempeño de la combinación seleccionada corresponde a la evaluación final en `test`, fuera de R6.
- Los IC 95 % reflejan la incertidumbre por la muestra de `validation` (200 imágenes, 22 malignas): son amplios y se superponen entre las dos primeras combinaciones. La diferencia de AP media entre ellas es muy pequeña frente a la desviación estándar entre semillas (sección 8.2), de modo que la selección por AP es una aplicación de la regla y no evidencia de superioridad.
- La sensibilidad y la especificidad al umbral 0.5 dependen de la calibración de cada modelo; ConvNeXt-Tiny-A muestra una variabilidad grande de la sensibilidad entre semillas (véanse las matrices de confusión y la sección 8.4). El análisis a sensibilidades fijas es exploratorio y post hoc, y no interviene en la selección.
- Si la combinación seleccionada utilizara la condición B, heredaría las limitaciones documentadas en R5 (ROI de `train` generadas por un segmentador ajustado con esas mismas imágenes; dependencia de la calidad de la segmentación). La seleccionada utiliza la condición A.
- La condición C se excluyó por no ser operativa; sus resultados en R5 solo sirven como referencia idealizada.